# CellHit drug rankings

Merge GDSC and PRISM predictions, then rank drugs for each MSI/TMB cohort.

In [ ]:
import os
from pathlib import Path

project_root = os.environ.get("MSS_TMB_PROJECT_ROOT")
if project_root:
    PROJECT_ROOT = Path(project_root).expanduser().resolve()
else:
    cwd = Path.cwd()
    PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / "data").is_dir() and (p / "scripts").is_dir()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Set MSS_TMB_PROJECT_ROOT to the project directory.")


from pathlib import Path
import os, re, json, time, hashlib, sqlite3, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

BASE_DIR = Path(os.environ.get('CELLHIT_BASE_DIR', str(PROJECT_ROOT / 'data'))).expanduser()
CELLHIT_DIR = BASE_DIR / 'CellHit'
WEB_DIR = CELLHIT_DIR / 'results'

WEB_GDSC_FILES = sorted(WEB_DIR.glob('cellhit_gdsc*.csv'))
WEB_PRISM_FILES = sorted(WEB_DIR.glob('cellhit_prism*.csv'))
WEB_MIXED_FILES = sorted(WEB_DIR.glob('gdsc_prism*.csv'))

PREVIOUS_GDSC_MERGED_CACHE = (CELLHIT_DIR / 'cellhit_gdsc_rankings'
                              / 'cellhit_merged_predictions.sqlite')
if not PREVIOUS_GDSC_MERGED_CACHE.is_file():
    PREVIOUS_GDSC_MERGED_CACHE = (
        CELLHIT_DIR / "CRC_UCEC_MSI_TMB_RANKINGS_GDSC_WEB_MERGED" / "cellhit_merged_predictions.sqlite"
    )
ORIGINAL_MATCHED_CACHE = CELLHIT_DIR / 'CRC_UCEC_MSI_TMB_RANKINGS' / 'cellhit_matched_patients.sqlite'
ORIGINAL_INPUTS = {'GDSC':CELLHIT_DIR/'full_results_gdsc.csv',
                   'PRISM':CELLHIT_DIR/'full_results_prism.csv'}
CRC_CLINICAL = BASE_DIR/'clinical_data.csv'
UCEC_CLINICAL = BASE_DIR/'ucec_clinical.csv'

OUTPUT_DIR = CELLHIT_DIR / 'cellhit_gdsc_prism_rankings'
FIG_DIR = OUTPUT_DIR/'figures'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
FIG_DIR.mkdir(parents=True,exist_ok=True)
CACHE_DB = OUTPUT_DIR/'cellhit_merged_predictions.sqlite'
CACHE_MANIFEST = OUTPUT_DIR/'merged_cache_manifest.json'

CHUNK_SIZE = 100_000
REBUILD_CACHE = False
MIN_MODEL_CORR = 0.20
MIN_GROUP_COVERAGE = 0.80
MIN_GROUP_PATIENTS = 3
ALLOW_MISSING_MODEL_CORR = True
TOP_N_FIGURES = 10
TOP_N_PATIENT = 10
TOP_N_BOOTSTRAP = 15
BOOTSTRAP_REPS = 500
SEED = 42
DRUG_ALIASES = {'5-fluorouracil':'fluorouracil','5 fluorouracil':'fluorouracil',
                '5-fu':'fluorouracil','fluorouracil':'fluorouracil','taxol':'paclitaxel'}

if PREVIOUS_GDSC_MERGED_CACHE.is_file():
    BASE_CACHE = PREVIOUS_GDSC_MERGED_CACHE

    NEW_WEB_FILES = sorted(set(WEB_PRISM_FILES + WEB_MIXED_FILES))
else:
    BASE_CACHE = ORIGINAL_MATCHED_CACHE if ORIGINAL_MATCHED_CACHE.is_file() else None
    NEW_WEB_FILES = sorted(set(WEB_GDSC_FILES + WEB_PRISM_FILES + WEB_MIXED_FILES))

if not NEW_WEB_FILES:
    raise FileNotFoundError(f'No new cellhit_prism*.csv or gdsc_prism*.csv files in {WEB_DIR}')

print('Base matched predictions:', BASE_CACHE or 'original CSV fallback')
print('New web exports to add:')
for f in NEW_WEB_FILES:
    print(f'  {f.name} ({f.stat().st_size / 1e6:.1f} MB)')
print('Old files and caches will not be modified. New output:',OUTPUT_DIR)

## Read clinical MSI/TMB labels

Match predictions to clinical groups by TCGA patient barcode.

In [ ]:
GROUPS = ['MSI-H', 'MSS/TMB-H', 'MSS/TMB-L']
CANON = {'MSIH':'MSI-H', 'MSI-H':'MSI-H', 'MSS/TMB-H':'MSS/TMB-H',
         'MSS/TMB-L':'MSS/TMB-L'}

def patient_id(series):
    return series.astype('string').str.upper().str.strip().str.extract(
        r'^(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})(?:$|[-_.])', expand=False
    )

def read_clinical(fp, cancer):
    if not fp.is_file():
        raise FileNotFoundError(f'Missing clinical file: {fp}')
    header = pd.read_csv(fp, nrows=0).columns
    wanted = {'Tumor_Sample_Barcode','sample_submitter_id', 'case_submitter_id',
              'MSI_TMB', 'dataset', 'project_id'}
    use = [c for c in header if c in wanted]
    df = pd.read_csv(fp, usecols=use, low_memory=False)
    if 'MSI_TMB' not in df:
        raise ValueError(f'No MSI_TMB column in {fp}')
    barcode_col = next((c for c in ['Tumor_Sample_Barcode','sample_submitter_id','case_submitter_id']
                        if c in df), None)
    if barcode_col is None:
        raise ValueError(f'No TCGA barcode column in {fp}')
    if cancer == 'CRC' and 'dataset' in df:
        df = df[df['dataset'].astype(str).str.upper().isin(['TCGA-COAD','TCGA-READ'])].copy()
    if cancer == 'UCEC' and 'project_id' in df:
        df = df[df['project_id'].astype(str).str.upper().eq('TCGA-UCEC')].copy()
    df['patient_id'] = patient_id(df[barcode_col])
    df['group_label'] = df['MSI_TMB'].astype('string').str.strip().str.upper().map(CANON)
    df = df.dropna(subset=['patient_id','group_label'])
    conflicts = df.groupby('patient_id')['group_label'].nunique()
    if (conflicts > 1).any():
        raise ValueError(f'Conflicting group labels within {fp.name}: {conflicts[conflicts>1].index.tolist()[:10]}')
    return df[['patient_id','group_label']].drop_duplicates('patient_id').assign(cancer=cancer)

clinical = pd.concat([read_clinical(CRC_CLINICAL,'CRC'),
                      read_clinical(UCEC_CLINICAL,'UCEC')], ignore_index=True)
conflicting_cancers = clinical.groupby('patient_id')['cancer'].nunique()
if (conflicting_cancers > 1).any():
    raise ValueError('CRC/UCEC clinical entries overlap in TCGA patient ID')
clinical.to_csv(OUTPUT_DIR / 'clinical_patient_labels.csv',index=False)
print(pd.crosstab(clinical['cancer'],clinical['group_label']).reindex(columns=GROUPS, fill_value=0))
print('Distinct clinically labeled TCGA patients:',clinical.patient_id.nunique())
assert set(clinical.group_label).issubset(GROUPS)

## Build the prediction cache

Deduplicate repeated sample–drug rows. Average distinct aliquots within each patient and screening dataset.

In [ ]:
READ_COLS_OLD = ['DrugName','DrugID','Source','SampleIndex','Predictions','PredictionsStd',
                 'QuantileScore','ExperimentalMedian','ModelMSE','ModelCorr','PutativeTarget']
READ_COLS_WEB = ['DrugName','DrugID','index','prediction','std','QuantileScore','DrugMedian',
                 'PutativeTarget','dataset','Source']
CACHE_COLS = ['dataset','patient_id','cancer','group_label','drug_id','drug_name','drug_key',
              'target','prediction','pred_std','quantile_score','exp_median','model_mse',
              'model_corr','n_aliquots','prediction_origin']

def normalize_drug(name):
    s = re.sub(r'\s+',' ',str(name).strip().lower())
    return DRUG_ALIASES.get(s,s)

def file_signature(path):
    st = path.stat()
    return {'path':str(path.resolve()),'size':st.st_size,'mtime_ns':st.st_mtime_ns}

def get_manifest():
    inputs = NEW_WEB_FILES + [CRC_CLINICAL,UCEC_CLINICAL]
    if BASE_CACHE is not None:
        inputs.append(BASE_CACHE)
    else:
        inputs += list(ORIGINAL_INPUTS.values())
    return {'files':[file_signature(p) for p in inputs],
            'clinical_sha256':hashlib.sha256(clinical.sort_values('patient_id').to_csv(index=False).encode()).hexdigest(),
            'drug_aliases':DRUG_ALIASES, 'version':'both_sources_web_merge_v2_invalid_quantile_audit'}

def cache_valid():
    if REBUILD_CACHE or not CACHE_DB.is_file() or not CACHE_MANIFEST.is_file():return False
    try:
        with sqlite3.connect(CACHE_DB) as con:
            tables={r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")}
        return ('patient_predictions' in tables and
                json.loads(CACHE_MANIFEST.read_text()) == get_manifest())
    except Exception:return False

def init_tables(con):
    con.executescript('''
        CREATE TABLE clinical_labels(patient_id TEXT PRIMARY KEY,cancer TEXT,group_label TEXT);
        CREATE TABLE patient_predictions(
           dataset TEXT,patient_id TEXT,cancer TEXT,group_label TEXT,drug_id TEXT,
           drug_name TEXT,drug_key TEXT,target TEXT,prediction REAL,pred_std REAL,
           quantile_score REAL,exp_median REAL,model_mse REAL,model_corr REAL,
           n_aliquots INTEGER,prediction_origin TEXT);
    ''')
    clinical[['patient_id','cancer','group_label']].to_sql('clinical_labels',con,if_exists='append',index=False)

def load_from_existing_cache(con,path):
    with sqlite3.connect(path) as src:
        cols={r[1] for r in src.execute('PRAGMA table_info(patient_predictions)')}
    required=set(CACHE_COLS)
    if not required.issubset(cols):
        raise ValueError(f'Base cache {path} missing columns: {sorted(required-cols)}')
    con.execute('ATTACH DATABASE ? AS old_src',(str(path),))
    try:
        con.execute('''
          INSERT INTO patient_predictions
          SELECT p.dataset,p.patient_id,l.cancer,l.group_label,p.drug_id,p.drug_name,
                 p.drug_key,p.target,p.prediction,p.pred_std,p.quantile_score,p.exp_median,
                 p.model_mse,p.model_corr,p.n_aliquots,p.prediction_origin
          FROM old_src.patient_predictions p
          JOIN clinical_labels l ON p.patient_id=l.patient_id
          WHERE p.dataset IN ('GDSC','PRISM')
        ''')
        con.commit()
    finally:
        con.execute('DETACH DATABASE old_src')
    print('Loaded',f"{con.execute('SELECT COUNT(*) FROM patient_predictions').fetchone()[0]:,}",
          'existing patient–drug records without rescan')

def load_from_original_csv(con):

    total=0
    for db,path in ORIGINAL_INPUTS.items():
        if not path.is_file():raise FileNotFoundError(f'Missing original download: {path}')
        for ch in pd.read_csv(path,usecols=lambda c:c in READ_COLS_OLD,
                              dtype={'DrugID':'string','DrugName':'string','SampleIndex':'string'},
                              chunksize=CHUNK_SIZE,low_memory=False):
            needed={'DrugID','DrugName','SampleIndex','Predictions','QuantileScore'}
            if not needed.issubset(ch):raise ValueError(f'{path.name} lacks {needed-set(ch)}')
            if 'Source' in ch:ch=ch[ch.Source.astype('string').str.upper().eq('TCGA')]
            ch=ch[ch.SampleIndex.astype('string').str.match(r'^TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}(?:$|[-_.])',na=False)].copy()
            if ch.empty:continue
            ch['patient_id']=patient_id(ch.SampleIndex)
            ch=ch.merge(clinical,on='patient_id',how='inner')
            if ch.empty:continue
            idx=ch.index
            out=pd.DataFrame({'dataset':db,'patient_id':ch.patient_id,'cancer':ch.cancer,
              'group_label':ch.group_label,'drug_id':ch.DrugID.astype(str),
              'drug_name':ch.DrugName.astype(str),'drug_key':ch.DrugName.map(normalize_drug),
              'target':ch.get('PutativeTarget',pd.Series(index=idx,dtype=str)),
              'prediction':pd.to_numeric(ch.Predictions,errors='coerce'),
              'pred_std':pd.to_numeric(ch.get('PredictionsStd',pd.Series(index=idx,dtype=float)),errors='coerce'),
              'quantile_score':pd.to_numeric(ch.QuantileScore,errors='coerce'),
              'exp_median':pd.to_numeric(ch.get('ExperimentalMedian',pd.Series(index=idx,dtype=float)),errors='coerce'),
              'model_mse':pd.to_numeric(ch.get('ModelMSE',pd.Series(index=idx,dtype=float)),errors='coerce'),
              'model_corr':pd.to_numeric(ch.get('ModelCorr',pd.Series(index=idx,dtype=float)),errors='coerce'),
              'n_aliquots':1,'prediction_origin':'published_csv'})
            out=out.dropna(subset=['patient_id','drug_id','prediction','quantile_score'])
            if not out.empty:
                out[CACHE_COLS].to_sql('published_staging',con,index=False,if_exists='append')
                total+=len(out)
        print(f'Original {db} scan finished')
    if not total:raise RuntimeError('No original TCGA results matched the clinical cohorts')
    con.execute('''
      INSERT INTO patient_predictions
      SELECT dataset,patient_id,MIN(cancer),MIN(group_label),drug_id,
        MIN(drug_name),MIN(drug_key),MIN(target),AVG(prediction),AVG(pred_std),
        AVG(quantile_score),AVG(exp_median),AVG(model_mse),AVG(model_corr),
        COUNT(*),'published_csv'
      FROM published_staging GROUP BY dataset,patient_id,drug_id
    ''')
    con.execute('DROP TABLE published_staging')

def infer_dataset_from_filename(path):
    if path.name.startswith('cellhit_gdsc'):return 'GDSC'
    if path.name.startswith('cellhit_prism'):return 'PRISM'
    return None

def load_web_exports(con):
    audit=[]
    distinct_drugs={'GDSC':set(),'PRISM':set()}
    invalid_csv = OUTPUT_DIR / 'web_invalid_quantiles.csv'
    if invalid_csv.exists():
        invalid_csv.unlink()
    for path in NEW_WEB_FILES:
        n_scan=n_matched=n_invalid=n_roundoff=0
        per_db={'GDSC':0,'PRISM':0}
        file_score_min=float('inf')
        file_score_max=float('-inf')
        from_filename=infer_dataset_from_filename(path)
        for ch in pd.read_csv(path,usecols=lambda c:c in READ_COLS_WEB,
                              dtype={'DrugID':'string','DrugName':'string','index':'string','dataset':'string'},
                              chunksize=CHUNK_SIZE,low_memory=False):
            n_scan+=len(ch)
            needed={'DrugID','DrugName','index','prediction','QuantileScore'}
            if not needed.issubset(ch):raise ValueError(f'{path.name} missing {needed-set(ch)}')
            if 'dataset' in ch.columns:
                ds=ch['dataset'].astype('string').str.strip().str.upper()
                ds=ds.replace({'GDSC1':'GDSC','GDSC2':'GDSC'})
            elif from_filename:
                ds=pd.Series(from_filename,index=ch.index,dtype='string')
                warnings.warn(f'{path.name} has no dataset column, assuming {from_filename} based on its filename')
            else:
                raise ValueError(f'Ambiguous file {path.name} has no dataset column. Cannot separate GDSC from PRISM.')

            keep=ch['index'].astype('string').str.upper().str.match(
                r'^TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}(?:$|[-_.])',na=False)
            ch=ch.loc[keep].copy()
            if ch.empty:continue
            ch['dataset_standard']=ds.loc[ch.index]
            unknown=sorted(set(ch['dataset_standard'].dropna())-{'GDSC','PRISM'})
            if unknown or ch['dataset_standard'].isna().any():
                raise ValueError(f'Unexpected dataset values in {path.name}: {unknown}; '
                                 'check dataset column instead of mixing the drug libraries')
            if 'Source' in ch:
                ch=ch[ch['Source'].astype(str).str.upper().eq('TCGA')].copy()
            if ch.empty:continue
            ch['patient_id']=patient_id(ch['index'])
            ch=ch.merge(clinical,on='patient_id',how='inner')
            if ch.empty:continue
            idx=ch.index
            out=pd.DataFrame({
                'dataset':ch.dataset_standard.astype(str),'patient_id':ch.patient_id,
                'cancer':ch.cancer,'group_label':ch.group_label,
                'drug_id':ch.DrugID.astype(str),'drug_name':ch.DrugName.astype(str),
                'drug_key':ch.DrugName.map(normalize_drug),
                'target':ch.get('PutativeTarget',pd.Series(index=idx,dtype=str)),
                'sample_index':ch['index'].astype(str),
                'prediction':pd.to_numeric(ch.prediction,errors='coerce'),
                'pred_std':pd.to_numeric(ch.get('std',pd.Series(index=idx,dtype=float)),errors='coerce'),
                'quantile_score':pd.to_numeric(ch.QuantileScore,errors='coerce'),
                'exp_median':pd.to_numeric(ch.get('DrugMedian',pd.Series(index=idx,dtype=float)),errors='coerce'),
                'source_file':path.name,
                'raw_quantile_score':ch.QuantileScore.astype('string')})

            out=out.dropna(subset=['patient_id','drug_id','prediction']).copy()
            if out.empty:continue
            qs=out['quantile_score']
            finite=np.isfinite(qs)
            if finite.any():
                file_score_min=min(file_score_min,float(qs[finite].min()))
                file_score_max=max(file_score_max,float(qs[finite].max()))

            near_boundary = finite & (((qs<0)&(qs>=-1e-8)) |
                                      ((qs>1)&(qs<=1+1e-8)))
            n_roundoff+=int(near_boundary.sum())
            if near_boundary.any():
                out.loc[near_boundary,'quantile_score']=out.loc[near_boundary,'quantile_score'].clip(0,1)
            qs=out['quantile_score']
            invalid=(~np.isfinite(qs)) | (qs<0) | (qs>1)
            if invalid.any():
                bad=out.loc[invalid,['source_file','dataset','cancer','group_label',
                       'patient_id','sample_index','drug_id','drug_name',
                       'raw_quantile_score','quantile_score','prediction']].copy()
                bad['reason']=np.select(
                    [~np.isfinite(bad.quantile_score),bad.quantile_score<0,bad.quantile_score>1],
                    ['missing_or_nonfinite','negative','above_one'],default='unknown')
                bad.to_csv(invalid_csv,mode='a',index=False,header=not invalid_csv.exists())
                n_invalid+=int(invalid.sum())
                out=out.loc[~invalid].copy()
            if out.empty:continue
            for db,part in out.groupby('dataset'):
                per_db[db]+=len(part)
                distinct_drugs[db].update(part.drug_id.unique().tolist())

            out=out.drop(columns=['raw_quantile_score'])
            out.to_sql('web_staging',con,index=False,if_exists='append')
            n_matched+=len(out)
        audit.append({'file':path.name,'rows_scanned':n_scan,'rows_matching_clinical':n_matched,
                      'gdsc_matched_rows':per_db['GDSC'],'prism_matched_rows':per_db['PRISM'],
                      'invalid_quantile_rows':n_invalid,
                      'tiny_boundary_roundoff_corrected':n_roundoff,
                      'min_numeric_quantile_seen':file_score_min if np.isfinite(file_score_min) else np.nan,
                      'max_numeric_quantile_seen':file_score_max if np.isfinite(file_score_max) else np.nan})
        print(f'{path.name}: scanned {n_scan:,}; matched valid-score rows {n_matched:,}; '
              f'invalid scores excluded {n_invalid:,}; boundary rounding corrected {n_roundoff:,} '
              f'(GDSC={per_db["GDSC"]:,}, PRISM={per_db["PRISM"]:,})')
        if n_invalid:
            warnings.warn(f'{path.name}: {n_invalid:,} invalid QuantileScore entries were logged to '
                          f'{invalid_csv.name} and excluded from score-based analysis. '
                          'Review before interpreting the drug rankings.')
            if n_invalid / max(1,n_matched+n_invalid) > 0.10:
                warnings.warn(f'{path.name}: over 10% of matched patient–drug scores were invalid; '
                              'this may materially affect drug rankings.')
    pd.DataFrame(audit).to_csv(OUTPUT_DIR/'new_web_export_file_audit.csv',index=False)
    print('Score audit summary:')
    display(pd.DataFrame(audit)[['file','rows_matching_clinical','invalid_quantile_rows',
                                 'tiny_boundary_roundoff_corrected',
                                 'min_numeric_quantile_seen','max_numeric_quantile_seen']])
    for db,ids in distinct_drugs.items():
        print(f'{db}: {len(ids):,} distinct drugs in NEW web exports')
        if len(ids) and len(ids)<10:
            warnings.warn(f'{db} web export has fewer than 10 drugs — check that it was not filtered by DrugName')
    n=con.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='web_staging'").fetchone()
    if not n:raise RuntimeError('No web predictions matched the clinical cohorts. Check SampleIndex/index and export dataset.')

    # DrugID must unambiguously name the same molecule WITHIN each database.
    conflicts=pd.read_sql_query('''
      SELECT dataset,drug_id,COUNT(DISTINCT drug_key) AS n_distinct_drug_names,
             GROUP_CONCAT(DISTINCT drug_name) AS observed_names
      FROM web_staging GROUP BY dataset,drug_id HAVING COUNT(DISTINCT drug_key)>1
    ''',con)
    conflicts.to_csv(OUTPUT_DIR/'web_internal_drug_id_conflicts.csv',index=False)
    if not conflicts.empty:
        raise ValueError('Web files reuse DrugID for different compounds within the same dataset; '
                         'review WEB_INTERNAL_DRUG_ID_CONFLICTS.csv')

    dup=pd.read_sql_query('''
      SELECT dataset,COUNT(*) AS repeated_sample_drug_keys,
             SUM(n_rows-1) AS redundant_rows,
             MAX(max_qs-min_qs) AS max_quantile_disagreement,
             MAX(max_pred-min_pred) AS max_prediction_disagreement
      FROM (SELECT dataset,patient_id,drug_id,sample_index,
              COUNT(*) AS n_rows,MAX(quantile_score) AS max_qs,MIN(quantile_score) AS min_qs,
              MAX(prediction) AS max_pred,MIN(prediction) AS min_pred
            FROM web_staging GROUP BY dataset,patient_id,drug_id,sample_index HAVING COUNT(*)>1)
      GROUP BY dataset
    ''',con)
    dup.to_csv(OUTPUT_DIR/'web_export_duplicate_audit.csv',index=False)
    if not dup.empty:display(dup)
    con.execute('''
       CREATE TABLE web_sample_dedup AS
       SELECT dataset,patient_id,cancer,group_label,drug_id,drug_name,drug_key,target,
              sample_index,prediction,pred_std,quantile_score,exp_median,source_file
       FROM (SELECT *,ROW_NUMBER() OVER
         (PARTITION BY dataset,patient_id,drug_id,sample_index ORDER BY rowid DESC) AS rn
         FROM web_staging) WHERE rn=1
    ''')
    con.execute('''
       CREATE TABLE web_patient AS
       SELECT dataset,patient_id,MIN(cancer) AS cancer,MIN(group_label) AS group_label,
              drug_id,MIN(drug_name) AS drug_name,MIN(drug_key) AS drug_key,
              MIN(target) AS target,AVG(prediction) AS prediction,AVG(pred_std) AS pred_std,
              AVG(quantile_score) AS quantile_score,AVG(exp_median) AS exp_median,
              COUNT(*) AS n_aliquots
       FROM web_sample_dedup GROUP BY dataset,patient_id,drug_id
    ''')
    print('Unique web patient–drug rows:',f"{con.execute('SELECT COUNT(*) FROM web_patient').fetchone()[0]:,}")
    con.execute('CREATE INDEX idx_web_patient ON web_patient(dataset,patient_id,drug_id)')
    con.execute('DROP TABLE web_sample_dedup')
    con.execute('DROP TABLE web_staging')

def merge_web_over_existing(con):
    conflicts=pd.read_sql_query('''
       SELECT o.dataset,o.drug_id,o.drug_key AS existing_drug,w.drug_key AS web_drug
       FROM (SELECT dataset,drug_id,MIN(drug_key) AS drug_key FROM patient_predictions
             GROUP BY dataset,drug_id) o
       JOIN (SELECT dataset,drug_id,MIN(drug_key) AS drug_key FROM web_patient
             GROUP BY dataset,drug_id) w
         ON o.dataset=w.dataset AND o.drug_id=w.drug_id
       WHERE o.drug_key<>w.drug_key
    ''',con)
    conflicts.to_csv(OUTPUT_DIR/'existing_vs_web_drug_id_conflicts.csv',index=False)
    if not conflicts.empty:
        raise ValueError('DrugID maps to different compound names between existing and web data. '
                         'Inspect EXISTING_VS_WEB_DRUG_ID_CONFLICTS.csv before merging.')
    con.execute('CREATE INDEX idx_existing_pd ON patient_predictions(dataset,patient_id,drug_id)')
    overlap=pd.read_sql_query('''
      SELECT p.dataset,p.cancer,p.group_label,
         COUNT(*) AS n_overlap_patient_drugs,
         AVG(ABS(p.prediction-w.prediction)) AS mean_abs_prediction_change,
         AVG(ABS(p.quantile_score-w.quantile_score)) AS mean_abs_quantile_change
      FROM patient_predictions p JOIN web_patient w
      ON p.dataset=w.dataset AND p.patient_id=w.patient_id AND p.drug_id=w.drug_id
      GROUP BY p.dataset,p.cancer,p.group_label
    ''',con)
    overlap.to_csv(OUTPUT_DIR/'overlapping_existing_vs_web.csv',index=False)
    if not overlap.empty:display(overlap)
    con.execute('''
      DELETE FROM patient_predictions
      WHERE EXISTS(SELECT 1 FROM web_patient w
         WHERE w.dataset=patient_predictions.dataset
           AND w.patient_id=patient_predictions.patient_id
           AND w.drug_id=patient_predictions.drug_id)
    ''')
    con.execute('''
      INSERT INTO patient_predictions
      SELECT dataset,patient_id,cancer,group_label,drug_id,drug_name,drug_key,target,
             prediction,pred_std,quantile_score,exp_median,NULL,NULL,n_aliquots,'new_web_export'
      FROM web_patient
    ''')
    con.execute('CREATE UNIQUE INDEX idx_unique_db_patient_drug ON patient_predictions(dataset,patient_id,drug_id)')
    con.execute('CREATE INDEX idx_dataset_group_drug ON patient_predictions(dataset,cancer,group_label,drug_id)')
    con.execute('CREATE INDEX idx_dataset_patient ON patient_predictions(dataset,patient_id)')
    con.execute('DROP TABLE web_patient')
    con.commit()

if cache_valid():
    print('Reusing previously completed updated cache:',CACHE_DB)
else:
    print('Building GDSC + PRISM merged cache...')
    t0=time.time()
    if CACHE_DB.exists(): CACHE_DB.unlink()
    with sqlite3.connect(CACHE_DB) as con:
        con.execute('PRAGMA journal_mode=DELETE')
        con.execute('PRAGMA synchronous=NORMAL')
        init_tables(con)
        if BASE_CACHE is not None: load_from_existing_cache(con,BASE_CACHE)
        else: load_from_original_csv(con)
        before=pd.read_sql_query('''SELECT dataset,cancer,group_label,
              COUNT(DISTINCT patient_id) AS n_patients FROM patient_predictions
              GROUP BY dataset,cancer,group_label''',con)
        before['version']='before'
        load_web_exports(con)
        merge_web_over_existing(con)
        after=pd.read_sql_query('''SELECT dataset,cancer,group_label,
              COUNT(DISTINCT patient_id) AS n_patients FROM patient_predictions
              GROUP BY dataset,cancer,group_label''',con)
        after['version']='after'
        diff=before.merge(after,on=['dataset','cancer','group_label'],how='outer',suffixes=('_before','_after')).fillna({'n_patients_before':0,'n_patients_after':0})
        diff['new_patients']=diff.n_patients_after-diff.n_patients_before
        diff.to_csv(OUTPUT_DIR/'patient_coverage_gains.csv',index=False)
        pd.concat([before,after],ignore_index=True).to_csv(OUTPUT_DIR/'patient_coverage_before_after.csv',index=False)
        print('Coverage gains:')
        display(diff.sort_values(['cancer','group_label','dataset']))
    CACHE_MANIFEST.write_text(json.dumps(get_manifest(),sort_keys=True,indent=2))
    print('Cache built in',round(time.time()-t0,1),'seconds')

with sqlite3.connect(CACHE_DB) as con:
    availability=pd.read_sql_query('''
      SELECT dataset,cancer,group_label,COUNT(DISTINCT patient_id) AS n_patients,
        COUNT(DISTINCT drug_id) AS n_drugs,COUNT(*) AS n_patient_drug_predictions,
        SUM(CASE WHEN n_aliquots>1 THEN 1 ELSE 0 END) AS n_multi_aliquot
      FROM patient_predictions GROUP BY dataset,cancer,group_label
    ''',con)
    origins=pd.read_sql_query('''
      SELECT dataset,prediction_origin,COUNT(*) AS n_patient_drug_predictions,
        COUNT(DISTINCT patient_id) AS n_patients
      FROM patient_predictions GROUP BY dataset,prediction_origin
    ''',con)
    overlap_counts=pd.read_sql_query('''
      SELECT cancer,group_label,
        COUNT(DISTINCT CASE WHEN n_databases=2 THEN patient_id END) AS patients_in_both_databases
      FROM (SELECT cancer,group_label,patient_id,COUNT(DISTINCT dataset) AS n_databases
            FROM patient_predictions GROUP BY cancer,group_label,patient_id)
      GROUP BY cancer,group_label
    ''',con)
    availability.to_csv(OUTPUT_DIR/'matched_samples_and_drugs.csv',index=False)
    origins.to_csv(OUTPUT_DIR/'prediction_source_breakdown.csv',index=False)
    overlap_counts.to_csv(OUTPUT_DIR/'patients_present_in_both_databases.csv',index=False)
print('Updated coverage by database and tumor group:')
display(availability)
print('Predictions retained by origin:')
display(origins)
print('Patients with predictions in BOTH GDSC and PRISM:')
display(overlap_counts)

## Apply assay quality filters

Require at least three patients, at least 80% coverage among matched patients, and model correlation of at least 0.20 when available.

In [ ]:
def iterate_drug_blocks(con, dataset, cancer, group, chunksize=80_000):
    sql = """SELECT drug_id,drug_name,drug_key,target,patient_id,prediction,pred_std,
                    quantile_score,exp_median,model_mse,model_corr,n_aliquots,prediction_origin
             FROM patient_predictions
             WHERE dataset=? AND cancer=? AND group_label=?
             ORDER BY drug_id,patient_id"""
    leftover=pd.DataFrame()
    for ch in pd.read_sql_query(sql,con,params=(dataset,cancer,group),chunksize=chunksize):
        if not leftover.empty: ch=pd.concat([leftover,ch],ignore_index=True)
        last=ch.drug_id.iloc[-1]
        complete,leftover=ch.loc[ch.drug_id!=last],ch.loc[ch.drug_id==last].copy()
        for _,block in complete.groupby('drug_id',sort=False): yield block
    if not leftover.empty:
        for _,block in leftover.groupby('drug_id',sort=False): yield block

def first_text(s):
    ss=s.dropna().astype(str)
    return ss.iloc[0] if len(ss) else ''

matched_group_counts = {(str(r.dataset),str(r.cancer),str(r.group_label)):int(r.n_patients)
                        for r in availability.itertuples(index=False)}
clinical_group_counts = {(c,g):int(((clinical.cancer==c)&(clinical.group_label==g)).sum())
                         for c in ['CRC','UCEC'] for g in GROUPS}
coverage=[]
for cancer in ['CRC','UCEC']:
    for group in GROUPS:
        n_clinical=clinical_group_counts[(cancer,group)]
        for dataset in ['GDSC','PRISM']:
            matched=matched_group_counts.get((dataset,cancer,group),0)
            coverage.append(dict(dataset=dataset,cancer=cancer,group_label=group,
                clinical_patients=n_clinical,matched_cellhit_patients=matched,
                fraction_of_original_cohort=matched/n_clinical if n_clinical else np.nan,
                low_n_warning=matched<10))
match_qc=pd.DataFrame(coverage)
match_qc.to_csv(OUTPUT_DIR/'clinical_to_cellhit_match_coverage.csv',index=False)
display(match_qc)

def drug_stat(df,dataset,cancer,group,n_total,n_clinical):
    vals=pd.to_numeric(df.quantile_score,errors='coerce').dropna()
    pred=pd.to_numeric(df.prediction,errors='coerce')
    m_corr=pd.to_numeric(df.model_corr,errors='coerce').dropna()
    m_mse=pd.to_numeric(df.model_mse,errors='coerce').dropna()
    n=df.patient_id.nunique()
    corr=float(m_corr.median()) if len(m_corr) else np.nan
    coverage=n/n_total if n_total else 0
    meets_quality=(not np.isfinite(corr) and ALLOW_MISSING_MODEL_CORR) or (np.isfinite(corr) and corr>=MIN_MODEL_CORR)
    return dict(dataset=dataset,cancer=cancer,group_label=group,drug_id=str(df.drug_id.iloc[0]),
        drug_name=first_text(df.drug_name),drug_key=first_text(df.drug_key),
        putative_target=first_text(df.target),n_patients=n,n_group_total=n_total,
        n_clinical_group_total=n_clinical,coverage=coverage,
        fraction_of_clinical_cohort_matched=n_total/n_clinical if n_clinical else np.nan,
        coverage_of_original_clinical_group=n/n_clinical if n_clinical else np.nan,
        limited_cohort_warning=n<10,
        mean_quantile_score=float(vals.mean()) if len(vals) else np.nan,
        median_quantile_score=float(vals.median()) if len(vals) else np.nan,
        q25_quantile_score=float(vals.quantile(.25)) if len(vals) else np.nan,
        q75_quantile_score=float(vals.quantile(.75)) if len(vals) else np.nan,
        frac_patient_qs_ge_0_75=float((vals>=.75).mean()) if len(vals) else np.nan,
        mean_prediction=float(pred.mean()),
        mean_delta_vs_experimental_median=float((pred-pd.to_numeric(df.exp_median,errors='coerce')).mean()),
        median_prediction_std=float(pd.to_numeric(df.pred_std,errors='coerce').median()),
        model_corr=corr,model_mse=float(m_mse.median()) if len(m_mse) else np.nan,
        model_quality_unavailable=not len(m_corr),
        web_prediction_model_corr_unavailable=bool(df.prediction_origin.eq('new_web_export').any()),
        n_new_web_patient_drugs=int(df.prediction_origin.eq('new_web_export').sum()),
        n_published_patient_drugs=int(df.prediction_origin.ne('new_web_export').sum()),
        qc_pass=bool(n>=MIN_GROUP_PATIENTS and coverage>=MIN_GROUP_COVERAGE and len(vals)==len(df) and meets_quality))

stats=[]
with sqlite3.connect(CACHE_DB) as con:
    for cancer in ['CRC','UCEC']:
        for group in GROUPS:
            for dataset in ['GDSC','PRISM']:
                n_matched=matched_group_counts.get((dataset,cancer,group),0)
                for block in iterate_drug_blocks(con,dataset,cancer,group):
                    stats.append(drug_stat(block,dataset,cancer,group,n_matched,
                                           clinical_group_counts[(cancer,group)]))
per_assay_all=pd.DataFrame(stats)
if per_assay_all.empty: raise RuntimeError('No matched predictions: inspect prior notebook and input paths')
per_assay_all['qc_failure']=np.select(
  [per_assay_all.n_patients<MIN_GROUP_PATIENTS,
   per_assay_all.coverage<MIN_GROUP_COVERAGE,
   per_assay_all.mean_quantile_score.isna(),
   per_assay_all.model_corr.notna() & (per_assay_all.model_corr<MIN_MODEL_CORR),
   per_assay_all.model_quality_unavailable & (not ALLOW_MISSING_MODEL_CORR)],
  ['too_few_patients','insufficient_coverage','missing_quantile_score','low_model_corr','model_quality_missing'],
  default='passed')
per_assay_all.to_csv(OUTPUT_DIR/'all_assays_with_qc.csv',index=False)
display(per_assay_all.groupby(['dataset','cancer','group_label','qc_failure']).size().rename('n_drugs').reset_index())
per_assay=per_assay_all[per_assay_all.qc_pass].copy()
if per_assay.empty:raise RuntimeError('All drugs failed QC; inspect ALL_ASSAYS_WITH_QC.csv')

per_assay=per_assay.sort_values(['coverage','model_quality_unavailable','model_corr','drug_id'],
                                 ascending=[False,True,False,True])
per_assay[per_assay.duplicated(['dataset','cancer','group_label','drug_key'],keep=False)].to_csv(
    OUTPUT_DIR/'duplicate_drug_assays_review.csv',index=False)
per_assay=per_assay.drop_duplicates(['dataset','cancer','group_label','drug_key'],keep='first').copy()
per_assay['within_dataset_rank']=per_assay.groupby(['dataset','cancer','group_label']).mean_quantile_score.rank(
    method='min',ascending=False)
N=per_assay.groupby(['dataset','cancer','group_label']).drug_key.transform('size')
per_assay['within_dataset_rank_percentile']=np.where(N>1,(N-per_assay.within_dataset_rank)/(N-1),0.5)
per_assay=per_assay.sort_values(['cancer','group_label','dataset','within_dataset_rank'])
per_assay.to_csv(OUTPUT_DIR/'per_database_eligible_drug_rankings.csv',index=False)
print('Eligible drug counts:')
display(per_assay.groupby(['cancer','group_label','dataset']).size().rename('n_drugs').reset_index())

## Rank drugs by cohort

Average available within-dataset rank percentiles. Higher CellHit QuantileScore indicates greater predicted sensitivity.

In [ ]:
ranking_rows=[]
for (cancer,group,drug_key),dg in per_assay.groupby(['cancer','group_label','drug_key'],sort=False):
    names=dg.sort_values(['model_quality_unavailable','model_corr'],ascending=[True,False]).drug_name
    row=dict(cancer=cancer,group_label=group,drug_key=drug_key,drug_name=names.iloc[0],
        putative_target='; '.join(sorted(set(str(x) for x in dg.putative_target if pd.notna(x) and str(x).strip()))),
        datasets='+'.join(sorted(dg.dataset.unique())),n_databases=int(dg.dataset.nunique()),
        mean_of_within_dataset_rank_percentiles=float(dg.within_dataset_rank_percentile.mean()),
        minimum_within_dataset_rank_percentile=float(dg.within_dataset_rank_percentile.min()),
        median_model_corr=float(dg.model_corr.median()) if dg.model_corr.notna().any() else np.nan,
        min_matched_patients=int(dg.n_group_total.min()),
        clinical_group_patients=int(dg.n_clinical_group_total.max()),
        minimum_clinical_match_fraction=float(dg.fraction_of_clinical_cohort_matched.min()),
        limited_cohort_warning=bool(dg.limited_cohort_warning.any()),
        any_model_quality_unavailable=bool(dg.model_quality_unavailable.any()),
        any_web_prediction_model_corr_unavailable=bool(dg.web_prediction_model_corr_unavailable.any()),
        both_sources_top_quartile=bool(dg.dataset.nunique()==2 and (dg.within_dataset_rank_percentile>=.75).all()))
    for _,r in dg.iterrows():
        prefix=r.dataset.lower()+'_'
        for a,b in [('mean_quantile_score','mean_qs'),('median_quantile_score','median_qs'),
            ('within_dataset_rank','rank'),('within_dataset_rank_percentile','rank_percentile'),
            ('n_patients','n_patients'),('model_corr','model_corr'),('model_mse','model_mse'),
            ('median_prediction_std','median_prediction_std'),('mean_prediction','mean_prediction'),
            ('mean_delta_vs_experimental_median','delta_vs_experimental_median'),
            ('n_new_web_patient_drugs','new_web_patient_drugs'),
            ('n_published_patient_drugs','published_patient_drugs'),
            ('model_quality_unavailable','model_quality_unavailable'),
            ('web_prediction_model_corr_unavailable','web_prediction_model_corr_unavailable')]:
            row[prefix+b]=r[a]
        row[prefix+'drug_id']=r.drug_id
    ranking_rows.append(row)
combined=pd.DataFrame(ranking_rows)
combined=combined.sort_values(['cancer','group_label','mean_of_within_dataset_rank_percentiles',
                               'n_databases','median_model_corr'],ascending=[True,True,False,False,False]).copy()
combined['rank']=combined.groupby(['cancer','group_label']).cumcount()+1
for cancer in ['CRC','UCEC']:
    for group in GROUPS:
        sub=combined[(combined.cancer==cancer)&(combined.group_label==group)].copy()
        safe=group.replace('/','_').replace('-','_')
        out=OUTPUT_DIR/f'ranked_{cancer.lower()}_{safe.lower()}_cellhit.csv'
        sub.to_csv(out,index=False)
        print(f'\n{cancer} {group}: {len(sub)} eligible drugs -> {out.name}')
        display(sub[['rank','drug_name','datasets','mean_of_within_dataset_rank_percentiles',
                     'median_model_corr','any_web_prediction_model_corr_unavailable']].head(10))
combined.to_csv(OUTPUT_DIR/'all_six_rankings_cellhit.csv',index=False)

## Rank drugs for individual tumors

In [ ]:
eligible_models = per_assay[['dataset','cancer','group_label','drug_id']].drop_duplicates().copy()
with sqlite3.connect(CACHE_DB) as con:
    eligible_models.to_sql('eligible_models',con,if_exists='replace',index=False)
    con.execute('CREATE INDEX IF NOT EXISTS idx_eligible ON eligible_models(dataset,cancer,group_label,drug_id)')
    sql="""SELECT p.dataset,p.cancer,p.group_label,p.patient_id,p.drug_id,p.drug_name,
                  p.drug_key,p.target,p.quantile_score,p.prediction,p.pred_std,p.model_corr
           FROM patient_predictions p INNER JOIN eligible_models e
              ON p.dataset=e.dataset AND p.cancer=e.cancer AND
                 p.group_label=e.group_label AND p.drug_id=e.drug_id
           ORDER BY p.dataset,p.patient_id"""
    groups=[]
    leftover=pd.DataFrame()
    for ch in pd.read_sql_query(sql,con,chunksize=70_000):
        if not leftover.empty:
            ch=pd.concat([leftover,ch],ignore_index=True)
        keys=ch.dataset.astype(str)+'/'+ch.patient_id.astype(str)
        tailkey=keys.iloc[-1]
        tail=keys.eq(tailkey)
        ready, leftover=ch.loc[~tail],ch.loc[tail].copy()
        if not ready.empty:
            for _,block in ready.groupby(['dataset','patient_id'],sort=False):
                groups.append(block.nlargest(TOP_N_PATIENT,'quantile_score'))
    if not leftover.empty:
        for _,block in leftover.groupby(['dataset','patient_id'],sort=False):
            groups.append(block.nlargest(TOP_N_PATIENT,'quantile_score'))

patient_top=pd.concat(groups,ignore_index=True) if groups else pd.DataFrame()
if not patient_top.empty:
    patient_top=patient_top.sort_values(['cancer','group_label','dataset','patient_id','quantile_score'],
                                       ascending=[True,True,True,True,False])
    patient_top['patient_rank']=patient_top.groupby(['dataset','patient_id']).cumcount()+1
patient_top.to_csv(OUTPUT_DIR / 'patient_level_top_drugs_by_database.csv',index=False)
print('Patient-level top drug rows:',len(patient_top))

## Estimate patient-bootstrap intervals

Resample patients within each cohort to estimate 95% intervals for mean QuantileScore.

In [ ]:
rng=np.random.default_rng(SEED)
bootstrap_rows=[]
with sqlite3.connect(CACHE_DB) as con:
    for cancer in ['CRC','UCEC']:
        for group in GROUPS:
            chosen=combined[(combined.cancer==cancer)&(combined.group_label==group)].head(TOP_N_BOOTSTRAP)
            for _,r in chosen.iterrows():
                for dataset in ['GDSC','PRISM']:
                    col=f'{dataset.lower()}_drug_id'
                    if col not in r or pd.isna(r[col]):
                        continue
                    df=pd.read_sql_query("""SELECT patient_id,quantile_score FROM patient_predictions
                                            WHERE cancer=? AND group_label=? AND dataset=? AND drug_id=?""",
                        con,params=(cancer,group,dataset,str(r[col])))
                    vals=df.quantile_score.dropna().to_numpy(float)
                    if len(vals)<2:
                        continue
                    means=np.array([rng.choice(vals,size=len(vals),replace=True).mean()
                                    for _ in range(BOOTSTRAP_REPS)])
                    bootstrap_rows.append({'cancer':cancer,'group_label':group,'dataset':dataset,
                      'drug_key':r.drug_key,'drug_name':r.drug_name,'n_patients':len(vals),
                      'mean_qs':np.mean(vals),'bootstrap_ci_low':np.quantile(means,0.025),
                      'bootstrap_ci_high':np.quantile(means,0.975)})
boot=pd.DataFrame(bootstrap_rows)
boot.to_csv(OUTPUT_DIR / 'top_drugs_patient_bootstrap.csv',index=False)
print('Bootstrap CI rows:',len(boot))

## Plot top-ranked drugs

In [ ]:
for cancer in ['CRC','UCEC']:
    for group in GROUPS:
        safe = group.replace('/','_').replace('-','_')
        top=combined[(combined.cancer==cancer)&(combined.group_label==group)].head(TOP_N_FIGURES).copy()
        if top.empty: continue
        names=top.drug_name.tolist()
        y=np.arange(len(top))
        fig,ax=plt.subplots(figsize=(11,max(4,0.52*len(top)+1.5)))
        for db,offset,sym in [('GDSC',-0.12,'o'),('PRISM',0.12,'s')]:
            col=f'{db.lower()}_mean_qs'
            if col in top:
                vals=pd.to_numeric(top[col],errors='coerce').to_numpy()
                mask=np.isfinite(vals)
                ax.scatter(vals[mask],y[mask]+offset,s=75,marker=sym,label=db,zorder=3)
        ax.set_yticks(y,names); ax.invert_yaxis()
        ax.set_xlim(0,1.05); ax.set_xlabel('Mean CellHit QuantileScore within each dataset (higher is favorable)')
        ax.set_title(f'{cancer} | {group} | top {len(top)} drugs')
        ax.grid(axis='x',alpha=0.2); ax.legend();fig.tight_layout()
        fig.savefig(FIG_DIR/f'{cancer.lower()}_{safe.lower()}_top_drugs.png',dpi=180)
        fig.savefig(FIG_DIR/f'{cancer.lower()}_{safe.lower()}_top_drugs.pdf')
        plt.close(fig)

with sqlite3.connect(CACHE_DB) as con:
    for cancer in ['CRC','UCEC']:
        for group in GROUPS:
            safe = group.replace('/','_').replace('-','_')
            top=combined[(combined.cancer==cancer)&(combined.group_label==group)].head(5)
            for dataset in ['GDSC','PRISM']:
                series=[]; labels=[]
                for _,r in top.iterrows():
                    idcol=dataset.lower()+'_drug_id'
                    if idcol not in r or pd.isna(r[idcol]): continue
                    dat=pd.read_sql_query("""SELECT quantile_score FROM patient_predictions
                        WHERE cancer=? AND group_label=? AND dataset=? AND drug_id=?""",con,
                        params=(cancer,group,dataset,str(r[idcol])))
                    if not dat.empty:
                        series.append(dat.quantile_score.dropna().to_numpy(float))
                        labels.append(str(r.drug_name))
                if not series: continue
                fig,ax=plt.subplots(figsize=(max(8,1.4*len(series)+3),5))
                ax.boxplot(series, showfliers=False)
                ax.set_xticks(range(1, len(labels) + 1), labels)
                rng_jitter=np.random.default_rng(SEED)
                for i, v in enumerate(series,1):
                    ax.scatter(i+rng_jitter.normal(0,0.06,len(v)),v,alpha=.27,s=13)
                ax.set_ylim(0,1);ax.set_ylabel('Patient-level CellHit QuantileScore')
                ax.set_title(f'{cancer} | {group} | {dataset}: top-drug distribution')
                ax.tick_params(axis='x',rotation=25);ax.grid(axis='y',alpha=.2)
                fig.tight_layout()
                fig.savefig(FIG_DIR/f'{cancer.lower()}_{safe.lower()}_{dataset.lower()}_top5_patient_distribution.png',dpi=180)
                fig.savefig(FIG_DIR/f'{cancer.lower()}_{safe.lower()}_{dataset.lower()}_top5_patient_distribution.pdf')
                plt.close(fig)
print('Figures saved to:',FIG_DIR)

## Summarize ranks for reference drugs

In [ ]:
CONTROLS = {
  'CRC': ['fluorouracil','oxaliplatin','irinotecan','cetuximab','regorafenib'],
  'UCEC': ['carboplatin','paclitaxel','cisplatin','doxorubicin','docetaxel','lenvatinib'],
}
control_rows=[]
for cancer,drug_names in CONTROLS.items():
    for group in GROUPS:
        sub=combined[(combined.cancer==cancer)&(combined.group_label==group)]
        for name in drug_names:
            found=sub[sub.drug_key==normalize_drug(name)]
            if found.empty:
                control_rows.append(dict(cancer=cancer,group_label=group,drug=name,rank=np.nan,
                                         status='not_in_QC_passed_rankings'))
            else:
                v=found.iloc[0]
                control_rows.append(dict(cancer=cancer,group_label=group,drug=name,
                       rank=v['rank'],datasets=v['datasets'],
                       relative_rank_score=v['mean_of_within_dataset_rank_percentiles'],status='ranked'))
controls=pd.DataFrame(control_rows)
controls.to_csv(OUTPUT_DIR / 'posthoc_known_drug_ranks.csv',index=False)
display(controls)